In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

def find_file(filename):
    candidates = [
        Path(filename),
        Path.cwd() / filename,
        Path("/mnt/data") / filename,
    ]
    for p in candidates:
        if p.exists():
            return p
    raise FileNotFoundError(f"Could not find {filename}")


try:
    details_path = find_file("order_details.csv")
except FileNotFoundError:
    details_path = find_file("order_details(3).csv")

df = pd.read_csv(details_path)

print("Dataset shape:", df.shape)
print("Cities:", sorted(df["order_city"].dropna().unique()))


Dataset shape: (151803, 10)
Cities: ['B', 'K', 'U']


In [2]:
city_bikers = (
    df.groupby("order_city")["biker"]
      .nunique()
      .rename("Unique Bikers")
      .reset_index()
      .rename(columns={"order_city": "City"})
      .sort_values("City")
)

display(city_bikers)


,City,Unique Bikers
0,B,478
1,K,2458
2,U,1362


In [3]:
district_vendors = (
    df.groupby(["order_city", "district"])["vendor"]
      .nunique()
      .rename("Unique Vendors")
      .reset_index()
      .rename(columns={
          "order_city": "City",
          "district": "District"
      })
      .sort_values(["City", "District"])
      .reset_index(drop=True)
)

display(district_vendors)


,City,District,Unique Vendors
0,B,3720,110
1,B,3724,39
2,B,3728,39
3,B,3732,15
4,B,3740,73
5,B,3752,5
6,K,1468,48
7,K,1472,1
8,K,1476,49
9,K,1480,299


In [4]:
summary = district_vendors.merge(
    city_bikers.rename(columns={"Unique Bikers": "Unique Bikers in City"}),
    on="City",
    how="left"
)

summary = summary[
    ["City", "District", "Unique Vendors", "Unique Bikers in City"]
]

display(summary)


,City,District,Unique Vendors,Unique Bikers in City
0,B,3720,110,478
1,B,3724,39,478
2,B,3728,39,478
3,B,3732,15,478
4,B,3740,73,478
5,B,3752,5,478
6,K,1468,48,2458
7,K,1472,1,2458
8,K,1476,49,2458
9,K,1480,299,2458


In [5]:
overall = pd.DataFrame({
    "Metric": [
        "Unique Cities",
        "Unique Districts",
        "Unique Vendors",
        "Unique Bikers"
    ],
    "Value": [
        df["order_city"].nunique(),
        df["district"].nunique(),
        df["vendor"].nunique(),
        df["biker"].nunique()
    ]
})

display(overall)


,Metric,Value
0,Unique Cities,3
1,Unique Districts,32
2,Unique Vendors,2110
3,Unique Bikers,4298
